In [ ]:
import pandas as pd
from pathlib import Path

# 1. Load the processed data
PROCESSED_DIR = Path('../Data/processed')
df_events = pd.read_csv(PROCESSED_DIR / 'events/master_events.csv')
df_clubs = pd.read_csv(PROCESSED_DIR / 'clubs/master_clubs.csv')
df_matches = pd.read_csv(PROCESSED_DIR / 'events/master_match_results.csv')

# 2. Helper Function: Extract a specific team's matches easily
def get_team_matches(team_name, match_df=df_matches):
    """Reshapes the data so the queried team is always the primary focus."""
    # When they were Team A
    as_a = match_df[match_df['Team_A_Name'] == team_name].copy()
    as_a['Target_Team'] = as_a['Team_A_Name']
    as_a['Opponent'] = as_a['Team_B_Name']
    as_a['Won_Match'] = as_a['Team_A_Won_Match']
    
    # When they were Team B
    as_b = match_df[match_df['Team_B_Name'] == team_name].copy()
    as_b['Target_Team'] = as_b['Team_B_Name']
    as_b['Opponent'] = as_b['Team_A_Name']
    as_b['Won_Match'] = as_b['Team_B_Won_Match']
    
    # Combine and return
    combined = pd.concat([as_a, as_b], ignore_index=True)
    return combined


In [ ]:
TARGET_CLUB = "The St. James Volleyball Club" # Replace with your club name

# Filter clubs, then merge with events to get the event details
club_events = df_clubs[df_clubs['Name'] == TARGET_CLUB]
club_event_details = pd.merge(club_events, df_events, on='EventId', how='inner')

print(f"{TARGET_CLUB} participated in {len(club_event_details)} events.")
display(club_event_details[['EventId', 'Name_y', 'StartDate', 'Location']].rename(columns={'Name_y': 'Event_Name'}))


In [ ]:
TARGET_TEAM = "The St. James Boys 16W" # Replace with your team name

# Use our helper function to get all matches for this team
team_matches = get_team_matches(TARGET_TEAM)

# Get unique EventIds and merge with the Events table
team_event_ids = team_matches['EventId'].drop_duplicates()
team_events = df_events[df_events['EventId'].isin(team_event_ids)]

print(f"{TARGET_TEAM} played in {len(team_events)} events.")
display(team_events[['EventId', 'Name', 'StartDate']])


In [ ]:
TARGET_TEAM = "The St. James Boys 16W"

team_matches = get_team_matches(TARGET_TEAM)

# Group by EventId and calculate total matches and wins
tournament_performance = team_matches.groupby('EventId').agg(
    Total_Matches=('Match_ID', 'count'),
    Matches_Won=('Won_Match', 'sum')
).reset_index()

# Calculate Losses and Win Percentage
tournament_performance['Matches_Lost'] = tournament_performance['Total_Matches'] - tournament_performance['Matches_Won']
tournament_performance['Win_Pct'] = (tournament_performance['Matches_Won'] / tournament_performance['Total_Matches'] * 100).round(1)

# Bring in the Event Name for readability
tournament_performance = pd.merge(tournament_performance, df_events[['EventId', 'Name']], on='EventId')

display(tournament_performance[['Name', 'Total_Matches', 'Matches_Won', 'Matches_Lost', 'Win_Pct']])


In [ ]:
TARGET_TEAM = "The St. James Boys 16W"
OPPONENT = "MVP 16 Black"

team_matches = get_team_matches(TARGET_TEAM)

# Filter for the specific opponent
head_to_head = team_matches[team_matches['Opponent'] == OPPONENT]

wins = head_to_head['Won_Match'].sum()
losses = len(head_to_head) - wins

print(f"Head-to-Head: {TARGET_TEAM} vs {OPPONENT}")
print(f"Record: {wins} Wins, {losses} Losses")
display(head_to_head[['EventId', 'Phase_Name', 'Time', 'Won_Match']])


In [ ]:
TARGET_TEAM = "The St. James Boys 16W"

# 1. Get all opponents that Team Y played against
team_matches = get_team_matches(TARGET_TEAM)
common_opponents = team_matches['Opponent'].unique()

print(f"Analyzing {len(common_opponents)} common opponents...\n")

# 2. Find ALL matches in the database where ANY team played against these common opponents
# We use the helper function in a loop for all common opponents
other_teams_data = []
for opp in common_opponents:
    opp_matches = get_team_matches(opp)
    
    # We only care about how OTHER teams did against this opponent (exclude Target Team)
    others_vs_opp = opp_matches[opp_matches['Opponent'] != TARGET_TEAM].copy()
    
    # In this context, 'Opponent' is the team playing against our Common Opponent. 
    # Let's rename for clarity: 
    # opp_matches['Target_Team'] = The Common Opponent
    # opp_matches['Opponent'] = The "Other" Team
    # opp_matches['Won_Match'] = Did the Common Opponent win? 
    # Therefore, Did the "Other" team win? = NOT Won_Match
    
    others_vs_opp['Other_Team_Name'] = others_vs_opp['Opponent']
    others_vs_opp['Other_Team_Won'] = ~others_vs_opp['Won_Match'] # The inverse
    
    other_teams_data.append(others_vs_opp)

# 3. Combine and Aggregate
df_others = pd.concat(other_teams_data)

comparison = df_others.groupby('Other_Team_Name').agg(
    Matches_vs_Common=('Match_ID', 'count'),
    Wins_vs_Common=('Other_Team_Won', 'sum')
).reset_index()

comparison['Win_Pct_vs_Common'] = (comparison['Wins_vs_Common'] / comparison['Matches_vs_Common'] * 100).round(1)

# Filter out teams that only played 1 match against common opponents to remove noise
comparison = comparison[comparison['Matches_vs_Common'] >= 3].sort_values(by='Win_Pct_vs_Common', ascending=False)

print(f"How other teams fared against {TARGET_TEAM}'s schedule (Min 3 matches):")
display(comparison.head(10))
